# 03 OR-Tools (CVRP)

Resolución del CVRP con OR-Tools usando VRP.csv.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from ortools.constraint_solver import pywrapcp, routing_enums_pb2
base = Path(r'd:\GitHub\Ruta_aprendizaje_2024\03-aplicación_de_modelos\proyecto_2\data')
df = pd.read_csv(base / 'VRP.csv')
df.columns = df.columns.str.lower()

## Construcción de matriz de distancia y demanda

In [ ]:
coords_ok = {'x','y'}.issubset(set(df.columns))
if coords_ok:
    pts = df[['x','y']].values.astype(float)
    dist_matrix = np.sqrt(((pts[:,None,:]-pts[None,:,:])**2).sum(-1)).astype(int)
else:
    dist_matrix = np.zeros((len(df), len(df)), dtype=int)
demand = df['demand'].astype(int) if 'demand' in df.columns else np.zeros(len(df), dtype=int)
capacity = int(df.get('capacity', pd.Series([1000])).iloc[0])
depot = int(df.get('depot', pd.Series([0])).iloc[0])
vehicles = int(df.get('vehicles', pd.Series([3])).iloc[0])

## Resolución con OR-Tools

In [ ]:
manager = pywrapcp.RoutingIndexManager(len(dist_matrix), vehicles, depot)
routing = pywrapcp.RoutingModel(manager)
def distance_callback(from_index, to_index):
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    return int(dist_matrix[from_node][to_node])
transit_callback_index = routing.RegisterTransitCallback(distance_callback)
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)
def demand_callback(from_index):
    from_node = manager.IndexToNode(from_index)
    return int(demand[from_node])
demand_callback_index = routing.RegisterUnaryTransitCallback(demand_callback)
routing.AddDimensionWithVehicleCapacity(demand_callback_index, 0, [capacity]*vehicles, True, 'Capacity')
search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
solution = routing.SolveWithParameters(search_parameters)
routes = []
total_cost = 0
if solution:
    for v in range(vehicles):
        index = routing.Start(v)
        route = []
        while not routing.IsEnd(index):
            route.append(manager.IndexToNode(index))
            previous_index = index
            index = solution.Value(routing.NextVar(index))
            total_cost += routing.GetArcCostForVehicle(previous_index, index, v)
        route.append(manager.IndexToNode(index))
        routes.append(route)
routes, total_cost